# ✈️ Carrier Insights

Analyze airline performance, market share, and pricing strategies.

## Setup

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_excel('airline_ticket_dataset.xlsx')
print(f"✅ Loaded {len(df):,} flight records")

## 1. Market Share by Large Carriers

Which airlines dominate the market?

In [ ]:
# Aggregate by large carrier
carrier_stats = df.groupby('carrier_lg').agg({
    'passengers': 'sum',
    'fare_lg': 'mean',
    'large_ms': 'mean'
}).reset_index()
carrier_stats.columns = ['Carrier', 'Total Passengers', 'Avg Fare', 'Market Share']

carrier_stats = carrier_stats.sort_values('Total Passengers', ascending=False)

fig = px.pie(carrier_stats, 
             values='Total Passengers', 
             names='Carrier',
             title='Market Share by Large Carriers (by Passenger Volume)',
             hole=0.4,
             hover_data={'Avg Fare': ':.2f'})

fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

## 2. Average Fare by Carrier

Which airlines are most/least expensive?

In [ ]:
carrier_sorted = carrier_stats.sort_values('Avg Fare', ascending=False)

fig = px.bar(carrier_sorted, 
             x='Carrier', 
             y='Avg Fare',
             title='Average Fare by Carrier (Large Carriers)',
             labels={'Avg Fare': 'Average Fare ($)', 'Carrier': 'Airline'},
             color='Avg Fare',
             color_continuous_scale='RdYlGn_r',
             text='Avg Fare')

fig.update_traces(texttemplate='$%{text:.2f}', textposition='outside')
fig.update_layout(showlegend=False)
fig.show()

## 3. Low-Cost Carrier Market Share

In [ ]:
lcc_stats = df.groupby('carrier_low').agg({
    'passengers': 'sum',
    'fare_low': 'mean',
    'lf_ms': 'mean'
}).reset_index()
lcc_stats.columns = ['Carrier', 'Total Passengers', 'Avg Fare', 'Market Share']

lcc_stats = lcc_stats.sort_values('Total Passengers', ascending=False)

fig = px.pie(lcc_stats, 
             values='Total Passengers', 
             names='Carrier',
             title='Market Share by Low-Cost Carriers',
             hole=0.4,
             color_discrete_sequence=px.colors.sequential.Teal)

fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

## 4. Carrier Pricing Comparison (Large vs Low-Cost)

In [ ]:
# Get all unique carriers
all_carriers = set(df['carrier_lg'].unique()) | set(df['carrier_low'].unique())

comparison_data = []
for carrier in all_carriers:
    lg_data = df[df['carrier_lg'] == carrier]['fare_lg']
    lc_data = df[df['carrier_low'] == carrier]['fare_low']
    
    if len(lg_data) > 0:
        comparison_data.append({
            'Carrier': carrier,
            'Type': 'Large Carrier',
            'Avg Fare': lg_data.mean(),
            'Routes': len(lg_data)
        })
    if len(lc_data) > 0:
        comparison_data.append({
            'Carrier': carrier,
            'Type': 'Low-Cost',
            'Avg Fare': lc_data.mean(),
            'Routes': len(lc_data)
        })

comparison_df = pd.DataFrame(comparison_data)

fig = px.bar(comparison_df, 
             x='Carrier', 
             y='Avg Fare',
             color='Type',
             barmode='group',
             title='Fare Comparison: Large Carriers vs Low-Cost Operations',
             labels={'Avg Fare': 'Average Fare ($)', 'Carrier': 'Airline'},
             hover_data={'Routes': True})

fig.update_layout(xaxis_tickangle=-45, height=600)
fig.show()

## 5. Carrier Performance: Passengers vs Pricing

In [ ]:
fig = px.scatter(carrier_stats, 
                 x='Avg Fare', 
                 y='Total Passengers',
                 size='Total Passengers',
                 color='Carrier',
                 text='Carrier',
                 title='Carrier Strategy: Price vs Volume',
                 labels={'Avg Fare': 'Average Fare ($)', 'Total Passengers': 'Total Passengers'},
                 size_max=60)

fig.update_traces(textposition='top center')
fig.update_layout(height=600, showlegend=False)
fig.show()

## 6. Market Share Distribution Analysis

In [ ]:
# Average market share by carrier
market_share_data = df.groupby('carrier_lg')['large_ms'].mean().sort_values(ascending=False).reset_index()
market_share_data.columns = ['Carrier', 'Average Market Share']

fig = px.bar(market_share_data, 
             x='Carrier', 
             y='Average Market Share',
             title='Average Market Share by Carrier',
             labels={'Average Market Share': 'Market Share (proportion)', 'Carrier': 'Airline'},
             color='Average Market Share',
             color_continuous_scale='Blues',
             text='Average Market Share')

fig.update_traces(texttemplate='%{text:.2%}', textposition='outside')
fig.update_layout(showlegend=False)
fig.show()

## 7. Carrier Route Coverage

In [ ]:
# Count unique routes per carrier
df['route'] = df['city1'] + ' → ' + df['city2']
route_coverage = df.groupby('carrier_lg')['route'].nunique().sort_values(ascending=False).reset_index()
route_coverage.columns = ['Carrier', 'Unique Routes']

fig = px.bar(route_coverage, 
             x='Carrier', 
             y='Unique Routes',
             title='Route Network Size by Carrier',
             labels={'Unique Routes': 'Number of Unique Routes', 'Carrier': 'Airline'},
             color='Unique Routes',
             color_continuous_scale='Greens',
             text='Unique Routes')

fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False)
fig.show()

## 8. Carrier Efficiency: Fare per Passenger

In [ ]:
# Calculate total revenue (fare * passengers) by carrier
carrier_revenue = df.groupby('carrier_lg').apply(
    lambda x: (x['fare_lg'] * x['passengers']).sum()
).reset_index()
carrier_revenue.columns = ['Carrier', 'Total Revenue']

# Merge with passenger totals
carrier_revenue = carrier_revenue.merge(
    carrier_stats[['Carrier', 'Total Passengers']], 
    on='Carrier'
)
carrier_revenue['Revenue per Passenger'] = carrier_revenue['Total Revenue'] / carrier_revenue['Total Passengers']

carrier_revenue = carrier_revenue.sort_values('Revenue per Passenger', ascending=False)

fig = px.bar(carrier_revenue, 
             x='Carrier', 
             y='Revenue per Passenger',
             title='Average Revenue per Passenger by Carrier',
             labels={'Revenue per Passenger': 'Revenue per Passenger ($)', 'Carrier': 'Airline'},
             color='Revenue per Passenger',
             color_continuous_scale='Oranges',
             text='Revenue per Passenger')

fig.update_traces(texttemplate='$%{text:.2f}', textposition='outside')
fig.update_layout(showlegend=False)
fig.show()

## 💡 Carrier Insights Summary

In [ ]:
print("="*60)
print("✈️ CARRIER INSIGHTS SUMMARY")
print("="*60)

# Market leader
market_leader = carrier_stats.iloc[0]
print(f"\n👑 Market Leader (by passengers):")
print(f"   • {market_leader['Carrier']}")
print(f"   • Total passengers: {market_leader['Total Passengers']:,.0f}")
print(f"   • Average fare: ${market_leader['Avg Fare']:.2f}")

# Most expensive
most_expensive = carrier_stats.nlargest(1, 'Avg Fare').iloc[0]
print(f"\n💎 Premium Carrier:")
print(f"   • {most_expensive['Carrier']}")
print(f"   • Average fare: ${most_expensive['Avg Fare']:.2f}")

# Budget option
budget = carrier_stats.nsmallest(1, 'Avg Fare').iloc[0]
print(f"\n💰 Most Affordable:")
print(f"   • {budget['Carrier']}")
print(f"   • Average fare: ${budget['Avg Fare']:.2f}")

# Widest network
widest_network = route_coverage.iloc[0]
print(f"\n🌐 Widest Network:")
print(f"   • {widest_network['Carrier']}")
print(f"   • Unique routes: {widest_network['Unique Routes']}")

print(f"\n📊 Overall Market:")
print(f"   • Total carriers (large): {len(carrier_stats)}")
print(f"   • Total carriers (low-cost): {len(lcc_stats)}")
print(f"   • Avg fare difference: ${carrier_stats['Avg Fare'].mean() - lcc_stats['Avg Fare'].mean():.2f}")

print("\n" + "="*60)